In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install ultralytics

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Search for 'valid' folder anywhere in your project
base = "/content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject"

print("🔍 Searching for all folders named 'valid' or 'train' or 'test'...\n")
for root, dirs, files in os.walk(base):
    for d in dirs:
        if d in ['valid', 'train', 'test', 'images']:
            full_path = os.path.join(root, d)
            try:
                count = len(os.listdir(full_path))
            except:
                count = "?"
            print(f"📁 {full_path}  →  {count} items")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Searching for all folders named 'valid' or 'train' or 'test'...

📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/test  →  2 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/train  →  2 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/valid  →  2 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/test/images  →  82 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/train/images  →  620 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/dataset/css-data/valid/images  →  0 items
📁 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test  →  2 items
📁 /content/driv

In [ ]:
yaml_path = "/content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/data.yaml"

content = """train: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/train/images
val: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/valid/images
test: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images
nc: 10
names: ["Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest", "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle"]
"""

with open(yaml_path, 'w') as f:
    f.write(content)

print("✅ data.yaml saved!")
with open(yaml_path, 'r') as f:
    print(f.read())

✅ data.yaml saved!
train: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/train/images
val: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/valid/images
test: /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images
nc: 10
names: ["Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest", "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle"]



In [ ]:

# Train
from ultralytics import YOLO
import torch

def train_on_cloud_gpu():
    # 1. Verify that the system sees your Tesla T4 GPU
    device = 0 if torch.cuda.is_available() else "cpu"
    print(f"Using device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

    # 2. Upgrade to the 'Small' model ('s') instead of 'Nano' ('n') to push accuracy past 90%
    model = YOLO("yolov8s.pt") 

    # 3. Optimized training for Tesla T4
    model.train(
        data= yaml_path,  # Replace with your actual path
        epochs=50,                      
        imgsz=640,                      # Keeps high resolution for small objects
        batch=32,                       # Increased to 32 (Tesla T4 handles this easily)
        workers=2,                      # Set to 2 to prevent Google Colab CPU memory crashes
        device=device,                  # Explicitly forces the code onto your Tesla T4 GPU
        
        # --- Hyperparameters tuned to maximize accuracy rapidly ---
        lr0=0.01,                       
        cos_lr=True,                    # Smoothly decays learning rate over 50 epochs
        box=7.5,                        # High focus on tight bounding boxes around PPE
        cls=1.2,                        # Strong class distinction
        overlap_mask=True,              # Helps when people are standing close together
        
        # --- Augmentations to prevent overfitting ---
        mosaic=1.0,                     # Mixes 4 images together (great for industrial scenes)
        mixup=0.15,    
        project="/content/drive/MyDrive/YOLO_Training",  
        name="safety_project"                                  
    )
    
    print("\nTraining complete! Running evaluation with optimized thresholds...")
    
    # 4. Evaluate performance with a tuned confidence threshold
    # Lowering this slightly allows the model to display valid detections it's slightly unsure of
    metrics = model.val(conf=0.20, iou=0.5)
    
    # Print the final score card
    print("\n=== FINAL METRICS ===")
    print(f"Precision (Accuracy of detections): {metrics.results_dict['metrics/precision(B)'] * 100:.2f}%")
    print(f"Recall (Percentage of total objects found): {metrics.results_dict['metrics/recall(B)'] * 100:.2f}%")
    print(f"mAP50 (Your main accuracy metric): {metrics.results_dict['metrics/mAP50(B)'] * 100:.2f}%")

if __name__ == "__main__": 
    train_on_cloud_gpu()

In [ ]:
import os
import pandas as pd

# Using the exact folder name from your screenshot with the hyphen
csv_path = (
    "/content/drive/MyDrive/YOLO_Training/safety_project-2/results.csv"
)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    last_row = df.iloc[-1]

    print("=== 📊 YOUR COMPLETED TRAINING METRICS ===")
    print(f"Epochs Completed: {int(last_row['epoch'])}/50")
    print(
        f"Precision Score:  {last_row['metrics/precision(B)'] * 100:.2f}%"
    )
    print(f"Recall Score:     {last_row['metrics/recall(B)'] * 100:.2f}%")
    print(f"mAP50 Score:      {last_row['metrics/mAP50(B)'] * 100:.2f}%")
else:
    print(
        f"❌ Google Colab still can't see the path. Please make sure you ran the drive.mount cell first today!"
    )

In [ ]:
import os
import torch
from ultralytics import YOLO


def extend_training_on_gpu():
    # 1. Double check that the notebook sees your Google GPU
    if torch.cuda.is_available():
        device = 0
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ GPU detected successfully! Training on: {gpu_name}")
    else:
        device = "cpu"
        print(
            "⚠️ WARNING: GPU not found. Please check your Colab Runtime settings."
        )

    # 2. Point to your 50-epoch weight checkpoint file inside your Google Drive
    checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project-2/weights/last.pt"

    # Quick safety fallback check in case the folder name doesn't have the '2'
    if not os.path.exists(checkpoint_path):
        checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project/weights/last.pt"

    print(f"🔄 Loading your existing progress from: {checkpoint_path}")
    model = YOLO(checkpoint_path)

    # 3. Start fine-tuning for epochs 51 to 100 to increase Recall & mAP50
    model.train(
        data=yaml_path,  # Path to your dataset configuration file
        epochs=100,  # Sets the final limit to 100 epochs total
        imgsz=640,  # Keeps high resolution for small safety objects
        batch=32,  # Batch size optimized for Google's Tesla GPU
        workers=2,  # Prevents CPU memory crashes on Colab
        device=device,  # Explicitly forces execution onto your GPU
        # --- Hyperparameters adjusted to catch missing objects ---
        lr0=0.002,  # Lower learning rate so it builds gently on your progress
        cos_lr=True,  # Smoothly decreases learning rate toward epoch 100
        box=8.5,  # Stronger focus on tight, accurate bounding boxes
        cls=1.5,  # Higher penalty for misclassifications to lift accuracy
        overlap_mask=True,
        # --- Heavy augmentations to maximize your current dataset ---
        mosaic=1.0,
        mixup=0.20,
        project="/content/drive/MyDrive/YOLO_Training",  # Your permanent Drive path
        name="safety_project",  # Saves automatically into a new folder (e.g., safety_project3)
    )

    print("\n🎉 Training to 100 epochs complete! Evaluating your new model...")

    # 4. Final validation with your mentor's target thresholds
    metrics = model.val(conf=0.20, iou=0.5)

    print("\n=== UPDATED 100-EPOCH SCORECARD ===")
    print(
        f"Precision (Detections accuracy): {metrics.results_dict['metrics/precision(B)'] * 100:.2f}%"
    )
    print(
        f"Recall (Percentage of objects found): {metrics.results_dict['metrics/recall(B)'] * 100:.2f}%"
    )
    print(
        f"mAP50 (Main accuracy metric): {metrics.results_dict['metrics/mAP50(B)'] * 100:.2f}%"
    )


if __name__ == "__main__":
    extend_training_on_gpu()

In [ ]:
import os
import torch
from ultralytics import YOLO


def extend_training_on_gpu():
    # 1. Double check that the notebook sees your Google GPU
    if torch.cuda.is_available():
        device = 0
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ GPU detected successfully! Training on: {gpu_name}")
    else:
        device = "cpu"
        print(
            "⚠️ WARNING: GPU not found. Please check your Colab Runtime settings."
        )

    # 2. Point to your 50-epoch weight checkpoint file inside your Google Drive
    checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project-3/weights/last.pt"

    # Quick safety fallback check in case the folder name doesn't have the '2'
    if not os.path.exists(checkpoint_path):
        checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project/weights/last.pt"

    print(f"🔄 Loading your existing progress from: {checkpoint_path}")
    model = YOLO(checkpoint_path)

    # 3. Start fine-tuning for epochs 51 to 100 to increase Recall & mAP50
    model.train(
        data=yaml_path,  # Path to your dataset configuration file
        epochs=100,  # Sets the final limit to 100 epochs total
        imgsz=640,  # Keeps high resolution for small safety objects
        batch=32,  # Batch size optimized for Google's Tesla GPU
        workers=2,  # Prevents CPU memory crashes on Colab
        device=device,  # Explicitly forces execution onto your GPU
        # --- Hyperparameters adjusted to catch missing objects ---
        lr0=0.002,  # Lower learning rate so it builds gently on your progress
        cos_lr=True,  # Smoothly decreases learning rate toward epoch 100
        box=8.5,  # Stronger focus on tight, accurate bounding boxes
        cls=1.5,  # Higher penalty for misclassifications to lift accuracy
        overlap_mask=True,
        # --- Heavy augmentations to maximize your current dataset ---
        mosaic=1.0,
        mixup=0.20,
        project="/content/drive/MyDrive/YOLO_Training",  # Your permanent Drive path
        name="safety_project",  # Saves automatically into a new folder (e.g., safety_project3)
    )

    print("\n🎉 Training to 100 epochs complete! Evaluating your new model...")

    # 4. Final validation with your mentor's target thresholds
    metrics = model.val(conf=0.20, iou=0.5)

    print("\n=== UPDATED 100-EPOCH SCORECARD ===")
    print(
        f"Precision (Detections accuracy): {metrics.results_dict['metrics/precision(B)'] * 100:.2f}%"
    )
    print(
        f"Recall (Percentage of objects found): {metrics.results_dict['metrics/recall(B)'] * 100:.2f}%"
    )
    print(
        f"mAP50 (Main accuracy metric): {metrics.results_dict['metrics/mAP50(B)'] * 100:.2f}%"
    )


if __name__ == "__main__":
    extend_training_on_gpu()

In [ ]:
import os
import torch
from ultralytics import YOLO


def extend_training_on_gpu():
    # 1. Double check that the notebook sees your Google GPU
    if torch.cuda.is_available():
        device = 0
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ GPU detected successfully! Training on: {gpu_name}")
    else:
        device = "cpu"
        print(
            "⚠️ WARNING: GPU not found. Please check your Colab Runtime settings."
        )

    # 2. Point to your 50-epoch weight checkpoint file inside your Google Drive
    checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project-4/weights/last.pt"

    # Quick safety fallback check in case the folder name doesn't have the '2'
    if not os.path.exists(checkpoint_path):
        checkpoint_path = "/content/drive/MyDrive/YOLO_Training/safety_project/weights/last.pt"

    print(f"🔄 Loading your existing progress from: {checkpoint_path}")
    model = YOLO(checkpoint_path)

    # 3. Start fine-tuning for epochs 51 to 100 to increase Recall & mAP50
    model.train(
        data=yaml_path,  # Path to your dataset configuration file
        epochs=1,  # Sets the final limit to 100 epochs total
        imgsz=640,  # Keeps high resolution for small safety objects
        batch=32,  # Batch size optimized for Google's Tesla GPU
        workers=2,  # Prevents CPU memory crashes on Colab
        device=device,  # Explicitly forces execution onto your GPU
        # --- Hyperparameters adjusted to catch missing objects ---
        lr0=0.002,  # Lower learning rate so it builds gently on your progress
        cos_lr=True,  # Smoothly decreases learning rate toward epoch 100
        box=8.5,  # Stronger focus on tight, accurate bounding boxes
        cls=1.5,  # Higher penalty for misclassifications to lift accuracy
        overlap_mask=True,
        # --- Heavy augmentations to maximize your current dataset ---
        mosaic=1.0,
        mixup=0.20,
        project="/content/drive/MyDrive/YOLO_Training",  # Your permanent Drive path
        name="safety_project",  # Saves automatically into a new folder (e.g., safety_project3)
    )

    print("\n🎉 Training to 100 epochs complete! Evaluating your new model...")

    # 4. Final validation with your mentor's target thresholds
    metrics = model.val(conf=0.20, iou=0.5)

    print("\n=== UPDATED 100-EPOCH SCORECARD ===")
    print(
        f"Precision (Detections accuracy): {metrics.results_dict['metrics/precision(B)'] * 100:.2f}%"
    )
    print(
        f"Recall (Percentage of objects found): {metrics.results_dict['metrics/recall(B)'] * 100:.2f}%"
    )
    print(
        f"mAP50 (Main accuracy metric): {metrics.results_dict['metrics/mAP50(B)'] * 100:.2f}%"
    )


if __name__ == "__main__":
    extend_training_on_gpu()

In [8]:
import os
import pandas as pd

# Using the exact folder name from your screenshot with the hyphen
csv_path = (
    "/content/drive/MyDrive/YOLO_Training/safety_project-5/results.csv"
)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    last_row = df.iloc[-1]

    print("=== 📊 YOUR COMPLETED TRAINING METRICS ===")
    print(f"Epochs Completed: {int(last_row['epoch'])}/102")
    print(
        f"Precision Score:  {last_row['metrics/precision(B)'] * 100:.2f}%"
    )
    print(f"Recall Score:     {last_row['metrics/recall(B)'] * 100:.2f}%")
    print(f"mAP50 Score:      {last_row['metrics/mAP50(B)'] * 100:.2f}%")
else:
    print(
        f"❌ Google Colab still can't see the path. Please make sure you ran the drive.mount cell first today!"
    )

=== 📊 YOUR COMPLETED TRAINING METRICS ===
Epochs Completed: 1/102
Precision Score:  91.40%
Recall Score:     78.03%
mAP50 Score:      84.29%


In [6]:
import os
import torch
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_Training/safety_project-5/weights/best.pt")

# 2. Force YOLO to look inside the hidden 'images' folder level
results = model.predict(source="/content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images", save=True, conf=0.25)

print("\n--- DONE! Check 'runs/detect/predict' for your labeled pictures! ---")



image 1/82 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images/-4405-_png_jpg.rf.82b5c10b2acd1cfaa24259ada8e599fe.jpg: 640x640 1 Hardhat, 3 NO-Safety Vests, 1 Person, 603.7ms
image 2/82 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images/000005_jpg.rf.96e9379ccae638140c4a90fc4b700a2b.jpg: 640x640 2 Hardhats, 2 NO-Masks, 3 Persons, 425.6ms
image 3/82 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images/002551_jpg.rf.ce4b9f934161faa72c80dc6898d37b2d.jpg: 640x640 2 Hardhats, 1 NO-Mask, 3 NO-Safety Vests, 3 Persons, 1 machinery, 420.4ms
image 4/82 /content/drive/MyDrive/safetyeyeinfosycollab/safetyeyeinfosysproject/datasetsafetyeye (Unzipped Files)/css-data/test/images/003357_jpg.rf.9867f91e88089bb68dc95947d5116d14.jpg: 640x640 1 Hardhat, 1 NO-Mask, 1 NO-Safety Vest, 1 Person, 